# Notebook 04 — Tree Models (LightGBM & CatBoost)

**Requires**:
- Notebook 02 outputs in `../processed_features/`
- Notebook 03 outputs (`nlp_*.npy`) in `../processed_features/`

**Strategy**:
1. Combine parsed dimension features from Notebook 02 with the NLP Ridge predictions from Notebook 03.
2. Extract rich text-derived numerical features (like unit presence flags, measurement extraction statistics).
3. Set up native categorical representation for `PRODUCT_TYPE_ID`.
4. Train two tree models using MAPE loss:
   - **LightGBM**: Optimizing `objective='mape'` with native categoricals.
   - **CatBoost**: Optimizing `loss_function='MAPE'` with native categoricals.
5. Save predictions to `../processed_features/lgb_*.npy` and `../processed_features/cb_*.npy`.

## 0. DATA MODE

In [1]:
# ============================================================
# DATA MODE: Changed to experiment mode
# ============================================================
DATA_MODE = "experiment"
DATA_PATHS = {
    "debug":      "../dataset/sampled/debug",
    "experiment": "../dataset/sampled/experiment",
    "full":       "../dataset"
}
DATA_DIR = DATA_PATHS[DATA_MODE]
print("Using dataset:", DATA_DIR)


Using dataset: ../dataset/sampled/experiment


## 1. Imports & Helpers

In [2]:
import os, re, numpy as np, pandas as pd
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool

def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = y_true > 0
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / y_true[mask]))

def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower().strip()
    text = re.sub(r'[\u00d7\u2715]', ' x ', text)
    text = re.sub(r'[^\x00-\x7f]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

print("✅ Imports & helpers done")


✅ Imports & helpers done


## 2. Load Data & Target

In [3]:
df_train_all = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
df_test      = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

train_idx = np.load("../processed_features/train_indices.npy")
val_idx   = np.load("../processed_features/val_indices.npy")

df_train = df_train_all.iloc[train_idx].reset_index(drop=True)
df_val   = df_train_all.iloc[val_idx].reset_index(drop=True)

y_train = df_train['PRODUCT_LENGTH'].values.astype(float)
y_val   = df_val['PRODUCT_LENGTH'].values.astype(float)

print(f"Train size: {len(df_train):,} | Val size: {len(df_val):,}")


Train size: 160,000 | Val size: 40,000


## 3. Build Rich Tabular Features

In [4]:
X_tr_df = pd.read_parquet("../processed_features/X_train_features.parquet")
X_va_df = pd.read_parquet("../processed_features/X_val_features.parquet")
X_te_df = pd.read_parquet("../processed_features/X_test_features.parquet")

nlp_tr = np.load("../processed_features/nlp_train.npy")
nlp_va = np.load("../processed_features/nlp_val.npy")
nlp_te = np.load("../processed_features/nlp_test.npy")

type_medians = df_train.groupby('PRODUCT_TYPE_ID')['PRODUCT_LENGTH'].median().to_dict()
type_means   = df_train.groupby('PRODUCT_TYPE_ID')['PRODUCT_LENGTH'].mean().to_dict()
type_stds    = df_train.groupby('PRODUCT_TYPE_ID')['PRODUCT_LENGTH'].std().fillna(0).to_dict()
type_counts  = df_train['PRODUCT_TYPE_ID'].value_counts().to_dict()
type_q25     = df_train.groupby('PRODUCT_TYPE_ID')['PRODUCT_LENGTH'].quantile(0.25).to_dict()
type_q75     = df_train.groupby('PRODUCT_TYPE_ID')['PRODUCT_LENGTH'].quantile(0.75).to_dict()
global_median = float(np.median(y_train))
global_std    = float(np.std(y_train))

UNIT_TO_PL = {
    'in':100.,'inch':100.,'inches':100.,'"':100.,
    'ft':1200.,'foot':1200.,'feet':1200.,
    'cm':39.37,'centimeter':39.37,'centimeters':39.37,
    'mm':3.937,'millimeter':3.937,'millimeters':3.937,
}

def extract_text_features(df):
    title   = df['TITLE'].fillna('').apply(clean_text)
    bullets = df['BULLET_POINTS'].fillna('').apply(clean_text)
    rows = []
    for i in range(len(df)):
        ti = title.iloc[i]
        bu = bullets.iloc[i]
        first_meas_title = 0.0
        for m in re.finditer(r'(\d+(?:[.,]\d+)?)\s*(in|inch|inches|"|cm|mm|ft|feet)', ti, re.I):
            v = float(m.group(1).replace(',','.')) * UNIT_TO_PL.get(m.group(2).lower(), 0)
            if 2 <= v <= 3_000_000:
                first_meas_title = v
                break
        rows.append({
            'first_meas_title': first_meas_title,
            'title_has_inch':   int(bool(re.search(r'\b(?:inch|inches|\d+")\b', ti))),
            'title_has_cm':     int(bool(re.search(r'\b\d+\s*cm\b', ti))),
            'title_has_mm':     int(bool(re.search(r'\b\d+\s*mm\b', ti))),
            'title_has_ft':     int(bool(re.search(r'\b(?:feet|foot|ft)\b', ti))),
            'title_has_x':      int(' x ' in ti),
            'title_num_count':  len(re.findall(r'\d+', ti)),
            'bullet_has_inch':  int(bool(re.search(r'\b(?:inch|inches)\b', bu))),
            'bullet_has_dim':   int(bool(re.search(r'\d+\s*(?:x|×)\s*\d+', bu))),
        })
    return pd.DataFrame(rows)

print("Extracting text features...")
tf_tr = extract_text_features(df_train)
tf_va = extract_text_features(df_val)
tf_te = extract_text_features(df_test)

DROP = {'PRODUCT_ID', 'PRODUCT_LENGTH'}
DIM_COLS = [c for c in X_tr_df.columns if c not in DROP]

def build_features(X_df, tf_df, df_src, nlp_preds):
    feat = pd.concat([
        X_df[DIM_COLS].fillna(0).reset_index(drop=True),
        tf_df.reset_index(drop=True)
    ], axis=1)
    feat = feat.loc[:, ~feat.columns.duplicated()]

    pid = df_src['PRODUCT_TYPE_ID'].reset_index(drop=True)
    feat['type_median']     = pid.map(type_medians).fillna(global_median)
    feat['type_mean']       = pid.map(type_means).fillna(global_median)
    feat['type_std']        = pid.map(type_stds).fillna(global_std)
    feat['type_count']      = pid.map(type_counts).fillna(0)
    feat['type_q25']        = pid.map(type_q25).fillna(global_median * 0.5)
    feat['type_q75']        = pid.map(type_q75).fillna(global_median * 1.5)
    feat['log_type_median'] = np.log1p(feat['type_median'])
    feat['type_iqr']        = feat['type_q75'] - feat['type_q25']

    eu = feat['explicit_length_u'].values
    tm = feat['type_median'].values
    feat['expl_to_type_ratio'] = np.where(eu > 0, eu / np.maximum(tm, 1), 1.0)
    feat['max_to_type_ratio']  = feat['max_dim_u'].values / np.maximum(tm, 1)
    fm = feat['first_meas_title'].values
    feat['first_meas_to_type'] = np.where(fm > 0, fm / np.maximum(tm, 1), 1.0)

    # Stack NLP Meta-feature
    feat['nlp_ridge_pred']     = nlp_preds
    feat['nlp_to_type_ratio']  = nlp_preds / np.maximum(tm, 1)

    feat['PRODUCT_TYPE_ID'] = pid.astype(int).values
    return feat

Xf_tr = build_features(X_tr_df, tf_tr, df_train, nlp_tr)
Xf_va = build_features(X_va_df, tf_va, df_val, nlp_va)
Xf_te = build_features(X_te_df, tf_te, df_test, nlp_te)

print(f"Feature Matrix shape: {Xf_tr.shape}")


Extracting text features...
Feature Matrix shape: (160000, 49)


## 4. Train LightGBM

In [5]:
CAT_COLS = ['PRODUCT_TYPE_ID']
dtrain = lgb.Dataset(Xf_tr, label=y_train, categorical_feature=CAT_COLS, free_raw_data=False)
dval   = lgb.Dataset(Xf_va, label=y_val, reference=dtrain, categorical_feature=CAT_COLS, free_raw_data=False)

params = {
    'objective':        'mape',
    'learning_rate':    0.03,
    'max_depth':        7,
    'num_leaves':       63,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'min_child_samples':10,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'seed':             42,
    'verbose':          -1
}

print("Training LightGBM (objective=mape)... ")
model_lgb = lgb.train(
    params, dtrain, num_boost_round=3000,
    valid_sets=[dval],
    callbacks=[
        lgb.early_stopping(100, verbose=False),
        lgb.log_evaluation(500)
    ]
)

lgb_tr = np.clip(model_lgb.predict(Xf_tr), 0.5, None)
lgb_va = np.clip(model_lgb.predict(Xf_va), 0.5, None)
lgb_te = np.clip(model_lgb.predict(Xf_te), 0.5, None)

print(f"LightGBM Train MAPE: {mape(y_train, lgb_tr)*100:.2f}%")
print(f"LightGBM Val MAPE:   {mape(y_val, lgb_va)*100:.2f}%")

np.save("../processed_features/lgb_train.npy", lgb_tr)
np.save("../processed_features/lgb_val.npy",   lgb_va)
np.save("../processed_features/lgb_test.npy",  lgb_te)


Training LightGBM (objective=mape)... 
LightGBM Train MAPE: 71.67%
LightGBM Val MAPE:   92.54%


## 5. Train CatBoost

In [6]:
train_pool = Pool(data=Xf_tr, label=y_train, cat_features=CAT_COLS)
val_pool   = Pool(data=Xf_va, label=y_val,   cat_features=CAT_COLS)
test_pool  = Pool(data=Xf_te, cat_features=CAT_COLS)

cb = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.05,
    depth=7,
    loss_function='MAPE',
    eval_metric='MAPE',
    random_seed=42,
    early_stopping_rounds=100,
    verbose=500
)

print("Training CatBoost (loss_function=MAPE)...")
cb.fit(train_pool, eval_set=val_pool)

cb_tr = np.clip(cb.predict(train_pool), 0.5, None)
cb_va = np.clip(cb.predict(val_pool), 0.5, None)
cb_te = np.clip(cb.predict(test_pool), 0.5, None)

print(f"CatBoost Train MAPE: {mape(y_train, cb_tr)*100:.2f}%")
print(f"CatBoost Val MAPE:   {mape(y_val, cb_va)*100:.2f}%")

np.save("../processed_features/cb_train.npy", cb_tr)
np.save("../processed_features/cb_val.npy",   cb_va)
np.save("../processed_features/cb_test.npy",  cb_te)

print("✅ Tree Predictions saved to processed_features/lgb_*.npy and cb_*.npy")


Training CatBoost (loss_function=MAPE)...
0:	learn: 0.9736187	test: 0.9806039	best: 0.9806039 (0)	total: 300ms	remaining: 10m
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9785053202
bestIteration = 19

Shrink model to first 20 iterations.
CatBoost Train MAPE: 95.52%
CatBoost Val MAPE:   97.85%
✅ Tree Predictions saved to processed_features/lgb_*.npy and cb_*.npy
